In [ ]:
# ============================================================
# MODEL COMPLEXITY COMPARISON (Proposed + 10 baselines)
# Params | Size | GFLOPs | GMACs | GPU & CPU Latency | Throughput | GPU Mem
# ============================================================
import os, time, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import timm
from torch.utils.flop_counter import FlopCounterMode

device  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUT_DIR = "/kaggle/working/output"
CKPT    = os.path.join(OUT_DIR, "DentAttn_MSFNet_final.pth")

IMG = 224                      # same input size for all (fair comparison)
NUM_CLASSES = 4
LAT_WARMUP, LAT_RUNS = 30, 100
CPU_RUNS = 30
TP_BATCH, TP_RUNS = 32, 20

BASELINES = {
    "ResNet18":          "resnet18",
    "ResNet50":          "resnet50",
    "VGG16":             "vgg16",
    "DenseNet121":       "densenet121",
    "InceptionV3":       "inception_v3",
    "EfficientNet-B0":   "efficientnet_b0",
    "MobileNetV3-Large": "mobilenetv3_large_100",
    "ConvNeXt-Tiny":     "convnext_tiny",
    "ViT-B/16":          "vit_base_patch16_224",
    "Swin-Tiny":         "swin_tiny_patch4_window7_224",
}

PROPOSED = "Proposed Model"

def get_model(name, tname):
    if name == PROPOSED:
        m, _ = load_proposed(CKPT, device="cpu")   # from your exported checkpoint
        return m
    return timm.create_model(tname, pretrained=False, num_classes=NUM_CLASSES)

# ---------- helpers ----------
def count_params(m):
    return (sum(p.numel() for p in m.parameters()),
            sum(p.numel() for p in m.parameters() if p.requires_grad))

def model_size_mb(m):
    s = sum(p.numel() * p.element_size() for p in m.parameters())
    s += sum(b.numel() * b.element_size() for b in m.buffers())
    return s / 1024**2

@torch.no_grad()
def count_flops(m, x):
    with FlopCounterMode(display=False) as fc:
        m(x)
    return fc.get_total_flops()          # 1 MAC = 2 FLOPs

def _sync(dev):
    if dev.type == "cuda":
        torch.cuda.synchronize()

@torch.no_grad()
def measure_latency(m, x, dev, warmup, runs):
    for _ in range(warmup):
        m(x)
    _sync(dev)
    ts = []
    for _ in range(runs):
        t0 = time.perf_counter()
        m(x)
        _sync(dev)
        ts.append((time.perf_counter() - t0) * 1000)
    return float(np.mean(ts)), float(np.std(ts))

@torch.no_grad()
def measure_throughput(m):
    xb = torch.randn(TP_BATCH, 3, IMG, IMG, device=device)
    for _ in range(5):
        m(xb)
    _sync(device)
    t0 = time.perf_counter()
    for _ in range(TP_RUNS):
        m(xb)
    _sync(device)
    return TP_BATCH * TP_RUNS / (time.perf_counter() - t0)

# ---------- run ----------
rows = []
for name, tname in {PROPOSED: None, **BASELINES}.items():
    print(f"▶ {name} ...", end=" ")
    model = None
    try:
        gc.collect()
        if device.type == "cuda":
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()

        model = get_model(name, tname).eval()
        total_p, train_p = count_params(model)
        size_mb = model_size_mb(model)

        # ---- CPU (FLOPs + CPU latency) ----
        model.to("cpu")
        x_cpu = torch.randn(1, 3, IMG, IMG)
        flops = count_flops(model, x_cpu)
        cpu_ms, _ = measure_latency(model, x_cpu, torch.device("cpu"), 5, CPU_RUNS)

        # ---- GPU (latency, throughput, memory) ----
        model.to(device)
        x1 = torch.randn(1, 3, IMG, IMG, device=device)
        lat, lat_std = measure_latency(model, x1, device, LAT_WARMUP, LAT_RUNS)
        try:
            fps = measure_throughput(model)
        except RuntimeError:
            fps = np.nan
        mem = torch.cuda.max_memory_allocated() / 1024**2 if device.type == "cuda" else np.nan

        rows.append({
            "Model": name,
            "Params (M)": total_p / 1e6,
            "Trainable (M)": train_p / 1e6,
            "Size (MB)": size_mb,
            "GFLOPs": flops / 1e9,
            "GMACs": flops / 2e9,
            "GPU Latency (ms)": lat,
            "GPU Lat Std": lat_std,
            "CPU Latency (ms)": cpu_ms,
            "Throughput (img/s)": fps,
            "Peak GPU Mem (MB)": mem,
        })
        print(f"✅ {total_p/1e6:.2f}M | {flops/1e9:.2f} GFLOPs | GPU {lat:.2f} ms | CPU {cpu_ms:.1f} ms")
    except Exception as e:
        print(f"❌ failed: {e}")
    finally:
        del model
        gc.collect()
        if device.type == "cuda":
            torch.cuda.empty_cache()

cx_df = pd.DataFrame(rows).round(3)
print("\n" + "=" * 110)
print(f"MODEL COMPLEXITY (input {IMG}x{IMG}, FP32, GPU={device})")
print("=" * 110)
print(cx_df.to_string(index=False))
cx_df.to_csv(os.path.join(OUT_DIR, "model_complexity.csv"), index=False)

# ---------- plots ----------
def bar(ax, col, title, unit):
    d = cx_df.sort_values(col)
    colors = ["crimson" if PROPOSED in m else "steelblue" for m in d["Model"]]
    ax.barh(d["Model"], d[col], color=colors, edgecolor="black")
    ax.set_title(title, fontweight="bold"); ax.set_xlabel(unit)
    ax.grid(True, axis="x", alpha=0.3)
    for i, v in enumerate(d[col]):
        ax.text(v, i, f" {v:.2f}", va="center", fontsize=9)

fig, axes = plt.subplots(2, 3, figsize=(22, 11))
bar(axes[0, 0], "Params (M)",       "Parameters",               "Millions")
bar(axes[0, 1], "GFLOPs",           "Computational Cost",       "GFLOPs")
bar(axes[0, 2], "Size (MB)",        "Model Size (FP32)",        "MB")
bar(axes[1, 0], "GPU Latency (ms)", "GPU Latency (batch=1)",    "ms")
bar(axes[1, 1], "CPU Latency (ms)", "CPU Latency (batch=1)",    "ms")
bar(axes[1, 2], "Throughput (img/s)", f"GPU Throughput (bs={TP_BATCH})", "img/s")
plt.suptitle("Model Complexity Comparison (Proposed in red)", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "model_complexity.png"), dpi=600, bbox_inches="tight")
plt.show()